# Week 12: GenAI for Data Science — Local Models, Evaluation & Error Analysis

## Learning Objectives

By the end of this session, you will be able to:
1. **Run and compare** open-source models locally with HuggingFace Transformers
2. **Build an evaluation framework** to measure LLM output quality (accuracy, consistency, format compliance)
3. **Perform systematic error analysis** on LLM outputs and iteratively improve prompts

## Prerequisites

- Completed Week 11 (LLM fundamentals, OpenAI API, basic HuggingFace)
- Watched pre-class videos on HuggingFace ecosystem, model selection, LLM evaluation metrics

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Recap | 5 min | Code |
| Section 1: HuggingFace Deep Dive | 30 min | Demo-heavy |
| Lab 1: HuggingFace Model Showdown | 15 min | Lab |
| Section 2: LLM Evaluation | 25 min | Demo + theory |
| Lab 2: Build an Evaluation Framework | 15 min | Lab |
| Section 3: Error Analysis | 15 min | Demo |
| Lab 3: Error Analysis & Prompt Iteration | 15 min | Lab |
| Wrap-up & Homework | 5 min | Markdown |

## What We'll Build Today

Building on Week 11's fraud detection task, we'll:
- Run multiple local models and compare their performance head-to-head
- Build a structured evaluation framework to score LLM outputs
- Use error analysis to systematically improve prompts through iteration

```
Today's Approach
================

  8 Fraud Transactions (from Week 11)
       |
  +----+-----------------------------+
  |              |                   |
  [GPT-2]    [Flan-T5]      [DistilBERT]    <-- Section 1: Local Models
  |              |                   |
  +----+-----------------------------+
       |
  [Evaluation Framework]                    <-- Section 2: Scoring & Metrics
  - Accuracy, Consistency
  - Format compliance
  - Hallucination detection
       |
  [Error Analysis Loop]                     <-- Section 3: Iterative Improvement
  - Categorize failures
  - Improve prompt -> re-run -> measure
```

## GPU Setup

**Runtime > Change runtime type > T4 GPU** (recommended but not required — all models run on CPU too)

# Section 0: Environment Setup

We'll use HuggingFace Transformers for local models and OpenAI for cloud API comparison.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# transformers: HuggingFace library for local model loading
# datasets: HuggingFace datasets library
# accelerate: Optimized model loading for HuggingFace
# openai: For cloud API comparison in homework
# evaluate: HuggingFace evaluation metrics library

!pip install -q transformers datasets torch accelerate openai evaluate

# =============================================================================
# IMPORTS
# =============================================================================
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    pipeline
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import json
import re
from collections import Counter

# =============================================================================
# VERIFY INSTALLATIONS
# =============================================================================
print("Library versions:")
print(f"  PyTorch:       {torch.__version__}")
print(f"  GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU device:    {torch.cuda.get_device_name(0)}")
print("\n✅ All libraries installed successfully!")

In [ ]:
# =============================================================================
# FRAUD TRANSACTION DATASET (same as Week 11)
# =============================================================================
# We reuse the same 8 synthetic transactions so you can compare
# local model performance against the cloud APIs from last week.

transaction_descriptions = [
    {
        "id": "TXN-001",
        "description": "Customer reports unauthorized wire transfer of $4,500 to unknown "
                       "overseas account. No prior international transaction history. "
                       "Transfer initiated at 3:47 AM local time.",
        "actual_label": "fraud"
    },
    {
        "id": "TXN-002",
        "description": "Regular monthly payment of $89.99 to Netflix streaming service. "
                       "Consistent with 18-month subscription history. Payment from "
                       "primary checking account.",
        "actual_label": "legitimate"
    },
    {
        "id": "TXN-003",
        "description": "Three consecutive ATM withdrawals totaling $1,500 in different "
                       "cities within 2 hours. Card was reported lost the following day. "
                       "Withdrawals at non-bank ATMs.",
        "actual_label": "fraud"
    },
    {
        "id": "TXN-004",
        "description": "Online purchase of $234.56 at Amazon.com for household electronics. "
                       "Shipping to address on file. Customer has frequent Amazon purchase "
                       "history.",
        "actual_label": "legitimate"
    },
    {
        "id": "TXN-005",
        "description": "Customer disputes charge of $2,100 at luxury jewelry store in Miami. "
                       "Customer's location confirmed as Chicago at time of purchase. No "
                       "travel alerts set.",
        "actual_label": "fraud"
    },
    {
        "id": "TXN-006",
        "description": "Automatic payroll direct deposit of $3,245.67 from employer ABC Corp. "
                       "Matches bi-weekly pay schedule. Amount consistent with employment "
                       "records.",
        "actual_label": "legitimate"
    },
    {
        "id": "TXN-007",
        "description": "Multiple small online purchases ($5-$15) at various digital stores "
                       "within 30 minutes. None of these merchants appear in customer's "
                       "history. Different IP addresses used.",
        "actual_label": "fraud"
    },
    {
        "id": "TXN-008",
        "description": "Grocery purchase of $67.23 at Whole Foods Market. Customer shops "
                       "here weekly based on 2-year transaction history. Paid with debit "
                       "card at POS terminal.",
        "actual_label": "legitimate"
    }
]

df = pd.DataFrame(transaction_descriptions)
print(f"Loaded {len(df)} transactions from Week 11 (4 fraud, 4 legitimate)")

# Section 1: HuggingFace Deep Dive — Running Models Locally

In Week 11, we saw two quick demos of HuggingFace models:
- **GPT-2**: Text completion with `pipeline("text-generation")`
- **Flan-T5**: Instruction-following with `AutoModelForSeq2SeqLM`

Today we go deeper. We'll explore **three different model architectures**
and see how they each approach the same fraud classification task:

| Model | Type | How It Works | Size |
|-------|------|-------------|------|
| **GPT-2** | Decoder-only | Completes text (like autocomplete) | 124M params |
| **Flan-T5-base** | Encoder-decoder | Follows task instructions | 250M params |
| **DistilBERT (SST-2)** | Encoder-only (fine-tuned) | Classifies sentiment directly | 67M params |

### Why Three Different Models?

Each architecture has different strengths:
- **GPT-2** generates free-form text — flexible but hard to control
- **Flan-T5** understands task prompts — good balance of control and flexibility
- **DistilBERT SST-2** is fine-tuned for one task — fast and consistent but inflexible

This mirrors real production decisions: do you want a general-purpose model
you can prompt, or a specialized model trained for your exact task?

In [ ]:
# =============================================================================
# DEMO: GPT-2 — Decoder-Only Text Completion
# =============================================================================
# GPT-2 doesn't "understand" tasks — it just continues text.
# We have to be clever with our prompt to get a classification.

generator = pipeline(
    "text-generation",
    model="gpt2",
    device=-1  # CPU
)

sample = transaction_descriptions[0]  # Unauthorized wire transfer

# Prompt engineered for text completion
prompt = f"""Transaction: "{sample['description']}"
Classification (fraud or legitimate):"""

start = time.time()
outputs = generator(
    prompt,
    max_new_tokens=5,
    temperature=0.01,
    do_sample=True,
    pad_token_id=50256  # Suppress warning
)
elapsed = time.time() - start

generated = outputs[0]['generated_text']
# Extract just the generated part (after our prompt)
prediction = generated[len(prompt):].strip().split()[0].lower().strip('.,!;:')

print(f"Transaction: {sample['id']}")
print(f"Full output: {generated}")
print(f"\nExtracted prediction: '{prediction}'")
print(f"Actual label:         '{sample['actual_label']}'")
print(f"Latency:              {elapsed:.2f}s")
print(f"\n💡 GPT-2 just continues text — it doesn't 'understand' classification.")
print(f"   The output might be 'fraud' or it might be random text!")

In [ ]:
# =============================================================================
# DEMO: Flan-T5 — Instruction-Following Encoder-Decoder
# =============================================================================
# Flan-T5 was fine-tuned on 1,000+ tasks. It understands prompts like
# "classify", "summarize", "translate" — much more controllable than GPT-2.

flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


def classify_with_flan(description, prompt_template):
    """Classify a transaction using Flan-T5 with a given prompt template."""
    prompt = prompt_template.format(description=description)
    inputs = flan_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    start = time.time()
    outputs = flan_model.generate(**inputs, max_new_tokens=20, do_sample=False)
    elapsed = time.time() - start
    result = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.strip().lower(), elapsed


# Zero-shot classification
sample = transaction_descriptions[0]
prompt_template = (
    "Classify the following bank transaction as 'fraud' or 'legitimate'.\n\n"
    "Transaction: \"{description}\"\n\n"
    "Classification:"
)

prediction, latency = classify_with_flan(sample['description'], prompt_template)

print(f"Transaction: {sample['id']}")
print(f"Prediction:  '{prediction}'")
print(f"Actual:      '{sample['actual_label']}'")
print(f"Latency:     {latency:.2f}s")
print(f"\n💡 Flan-T5 follows instructions! Much more predictable than GPT-2.")

In [ ]:
# =============================================================================
# DEMO: DistilBERT SST-2 — Fine-Tuned Sentiment Classifier
# =============================================================================
# This model was TRAINED specifically for sentiment classification.
# It outputs POSITIVE/NEGATIVE — not fraud/legitimate.
# But we can use sentiment as a proxy: fraud descriptions tend to be
# "negative" (unauthorized, stolen, suspicious) while legitimate ones
# are "neutral/positive" (regular, consistent, monthly).

classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1
)

sample = transaction_descriptions[0]

start = time.time()
result = classifier(sample['description'][:512])  # DistilBERT has 512 token limit
elapsed = time.time() - start

label = result[0]['label']    # POSITIVE or NEGATIVE
score = result[0]['score']

# Map sentiment to fraud prediction
fraud_prediction = "fraud" if label == "NEGATIVE" else "legitimate"

print(f"Transaction: {sample['id']}")
print(f"Sentiment:   {label} ({score:.3f})")
print(f"Mapped to:   '{fraud_prediction}'")
print(f"Actual:      '{sample['actual_label']}'")
print(f"Latency:     {elapsed:.3f}s")
print(f"\n💡 Fine-tuned models are FAST and consistent, but inflexible.")
print(f"   This model only knows POSITIVE/NEGATIVE — we're using it as a proxy.")
print(f"   A model fine-tuned on actual fraud data would be much better (Week 14).")

> **Think About It**: GPT-2 generated free text, Flan-T5 followed instructions,
> and DistilBERT gave a fixed classification. In production, which approach would
> you choose for fraud detection? What if you need to explain the decision to a
> regulator? What if you need to process 10,000 transactions per second?

In [ ]:
# =============================================================================
# DEMO: Head-to-Head Comparison — All Models, All Transactions
# =============================================================================

def classify_gpt2(description):
    """Classify with GPT-2 (text completion)."""
    prompt = f'Transaction: "{description}"\nClassification (fraud or legitimate):'
    out = generator(prompt, max_new_tokens=5, temperature=0.01,
                    do_sample=True, pad_token_id=50256)
    text = out[0]['generated_text'][len(prompt):].strip().split()[0].lower().strip('.,!;:')
    return text


def classify_flan(description):
    """Classify with Flan-T5 (instruction-following)."""
    prompt = (
        "Classify the following bank transaction as 'fraud' or 'legitimate'.\n\n"
        f'Transaction: "{description}"\n\nClassification:'
    )
    inputs = flan_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = flan_model.generate(**inputs, max_new_tokens=10, do_sample=False)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()


def classify_distilbert(description):
    """Classify with DistilBERT (sentiment as proxy)."""
    result = classifier(description[:512])
    return "fraud" if result[0]['label'] == "NEGATIVE" else "legitimate"


# Run all models on all transactions
results = []
for txn in transaction_descriptions:
    row = {'id': txn['id'], 'actual': txn['actual_label']}

    start = time.time()
    row['gpt2'] = classify_gpt2(txn['description'])
    row['gpt2_time'] = time.time() - start

    start = time.time()
    row['flan_t5'] = classify_flan(txn['description'])
    row['flan_t5_time'] = time.time() - start

    start = time.time()
    row['distilbert'] = classify_distilbert(txn['description'])
    row['distilbert_time'] = time.time() - start

    results.append(row)
    print(f"  ✓ {txn['id']}")

comparison_df = pd.DataFrame(results)
print(f"\n{'='*60}")
display(comparison_df[['id', 'actual', 'gpt2', 'flan_t5', 'distilbert']])

# Summary
for model in ['gpt2', 'flan_t5', 'distilbert']:
    acc = (comparison_df[model] == comparison_df['actual']).mean()
    avg_time = comparison_df[f'{model}_time'].mean()
    print(f"{model:12s}: accuracy={acc:.1%}, avg latency={avg_time:.3f}s")

## Lab 1: HuggingFace Model Showdown (15 minutes)

### Your Task

Extend the head-to-head comparison by adding **few-shot prompting** for Flan-T5
to see if providing examples improves the smaller model's accuracy.

### Steps

1. **Write a few-shot prompt template** for Flan-T5 with 2 examples
   (use different examples than the test transactions!)
2. **Create `classify_flan_fewshot(description)`** using your template
3. **Run it on all 8 transactions** and add results to the comparison DataFrame
4. **Create a bar chart** comparing accuracy across all 4 approaches
   (GPT-2, Flan-T5 zero-shot, Flan-T5 few-shot, DistilBERT)

### Expected Output

- Updated DataFrame with a `flan_t5_fewshot` column
- Bar chart showing accuracy per model
- Which approach won?

### Homework Extension

After class: add OpenAI GPT-4o-mini to the comparison (use your Week 11 code).
Create a full comparison table: accuracy, latency, estimated cost per call.

In [ ]:
# =============================================================================
# LAB 1: HUGGINGFACE MODEL SHOWDOWN
# =============================================================================

# YOUR CODE: Write a few-shot prompt template for Flan-T5
# Include 2 examples (NOT from the test transactions)
FEW_SHOT_TEMPLATE = None  # YOUR CODE


def classify_flan_fewshot(description):
    """Classify with Flan-T5 using few-shot prompting."""
    # YOUR CODE: Build prompt from template and classify
    prediction = None  # YOUR CODE
    return prediction


# YOUR CODE: Run few-shot Flan-T5 on all 8 transactions
fewshot_results = []  # YOUR CODE

# YOUR CODE: Add results to comparison_df
# comparison_df['flan_t5_fewshot'] = ...

# YOUR CODE: Create bar chart comparing accuracy of all 4 approaches
# fig, ax = plt.subplots(...)

# Verification
if 'flan_t5_fewshot' in comparison_df.columns:
    for model in ['gpt2', 'flan_t5', 'flan_t5_fewshot', 'distilbert']:
        acc = (comparison_df[model] == comparison_df['actual']).mean()
        print(f"{model:18s}: {acc:.1%}")
    print("\n🎉 Lab 1 complete!")
else:
    print("❌ Lab 1 incomplete — fill in the YOUR CODE sections above")

# Section 2: Evaluating LLM Outputs

Getting an LLM to generate a response is easy. Knowing whether that response
is **good** is hard. In production, you need systematic evaluation — not just
"it looks right."

## Why Evaluation Matters

| Without Evaluation | With Evaluation |
|---|---|
| "The model seems to work" | "The model has 87.5% accuracy on our test set" |
| "It sometimes gives weird answers" | "12% of outputs have format errors, 5% hallucinate" |
| "We updated the prompt and it's better now" | "Prompt v3 improved accuracy from 75% to 87.5% with no format regressions" |

## Four Dimensions of LLM Evaluation

| Dimension | What It Measures | How to Check |
|-----------|-----------------|--------------|
| **Accuracy** | Does it get the right answer? | Compare to ground truth labels |
| **Consistency** | Same input → same output? | Run N times, measure agreement |
| **Format Compliance** | Did it follow the output format? | Regex/parse check |
| **Hallucination** | Did it invent facts? | Check claims against source text |

In [ ]:
# =============================================================================
# DEMO: Generate Outputs to Evaluate
# =============================================================================
# We'll generate fraud classifications with Flan-T5 using a structured prompt
# that asks for label + reasoning. This gives us rich outputs to evaluate.

EVAL_PROMPT = """Analyze this bank transaction and respond in this exact format:
Label: <fraud or legitimate>
Reasoning: <one sentence explanation>
Confidence: <high, medium, or low>

Transaction: "{description}"
"""


def generate_for_eval(description):
    """Generate a structured response from Flan-T5 for evaluation."""
    prompt = EVAL_PROMPT.format(description=description)
    inputs = flan_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = flan_model.generate(**inputs, max_new_tokens=80, do_sample=False)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)


# Generate outputs for all transactions
eval_outputs = []
for txn in transaction_descriptions:
    raw_output = generate_for_eval(txn['description'])
    eval_outputs.append({
        'id': txn['id'],
        'actual_label': txn['actual_label'],
        'raw_output': raw_output
    })
    print(f"{txn['id']}: {raw_output[:80]}...")

eval_df = pd.DataFrame(eval_outputs)
print(f"\nGenerated {len(eval_df)} outputs for evaluation")

In [ ]:
# =============================================================================
# DEMO: Evaluation Dimension 1 — Accuracy
# =============================================================================
# Parse the label from each output and compare to ground truth.

def extract_label(raw_output):
    """Extract 'fraud' or 'legitimate' from model output."""
    output_lower = raw_output.lower()
    # Try to find "Label: fraud" or "Label: legitimate" pattern
    match = re.search(r'label:\s*(fraud|legitimate)', output_lower)
    if match:
        return match.group(1)
    # Fallback: check if fraud or legitimate appears anywhere
    if 'fraud' in output_lower:
        return 'fraud'
    if 'legitimate' in output_lower:
        return 'legitimate'
    return 'unparseable'


eval_df['predicted_label'] = eval_df['raw_output'].apply(extract_label)
eval_df['correct'] = eval_df['predicted_label'] == eval_df['actual_label']

accuracy = eval_df['correct'].mean()
print(f"Accuracy: {accuracy:.1%}")
print(f"Correct:  {eval_df['correct'].sum()}/{len(eval_df)}")
print()
display(eval_df[['id', 'actual_label', 'predicted_label', 'correct']])

# Show errors
errors = eval_df[~eval_df['correct']]
if len(errors) > 0:
    print(f"\n❌ Errors ({len(errors)}):")
    for _, row in errors.iterrows():
        print(f"  {row['id']}: predicted '{row['predicted_label']}', actual '{row['actual_label']}'")
        print(f"    Output: {row['raw_output'][:100]}")

In [ ]:
# =============================================================================
# DEMO: Evaluation Dimension 2 — Consistency
# =============================================================================
# Run the same prompt 5 times and check if we get the same answer.
# With do_sample=False (greedy), Flan-T5 should be perfectly consistent.
# But what happens with sampling enabled?

sample = transaction_descriptions[0]
N_RUNS = 5

# Greedy (deterministic)
greedy_results = []
for _ in range(N_RUNS):
    output = generate_for_eval(sample['description'])
    label = extract_label(output)
    greedy_results.append(label)

# With sampling (stochastic) — simulates what happens with temperature > 0
sampling_results = []
prompt = EVAL_PROMPT.format(description=sample['description'])
inputs = flan_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
for _ in range(N_RUNS):
    outputs = flan_model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.9)
    output = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    label = extract_label(output)
    sampling_results.append(label)

print(f"Transaction: {sample['id']} (actual: {sample['actual_label']})")
print(f"\nGreedy (do_sample=False):  {greedy_results}")
print(f"  Agreement: {max(Counter(greedy_results).values())}/{N_RUNS}")
print(f"\nSampling (temperature=0.9): {sampling_results}")
print(f"  Agreement: {max(Counter(sampling_results).values())}/{N_RUNS}")
print(f"\n💡 Greedy decoding = perfect consistency but potentially lower quality.")
print(f"   Sampling = more varied but inconsistent. Choose based on your use case.")

In [ ]:
# =============================================================================
# DEMO: Evaluation Dimension 3 — Format Compliance
# =============================================================================
# Did the model follow our requested format?
# Expected: "Label: ...\nReasoning: ...\nConfidence: ..."

def check_format_compliance(raw_output):
    """Check if output follows the expected Label/Reasoning/Confidence format."""
    checks = {
        'has_label': bool(re.search(r'label:', raw_output, re.IGNORECASE)),
        'has_reasoning': bool(re.search(r'reasoning:', raw_output, re.IGNORECASE)),
        'has_confidence': bool(re.search(r'confidence:', raw_output, re.IGNORECASE)),
        'label_valid': bool(re.search(r'label:\s*(fraud|legitimate)', raw_output, re.IGNORECASE)),
        'confidence_valid': bool(re.search(r'confidence:\s*(high|medium|low)', raw_output, re.IGNORECASE)),
    }
    checks['fully_compliant'] = all(checks.values())
    return checks


# Check all outputs
compliance_results = []
for _, row in eval_df.iterrows():
    checks = check_format_compliance(row['raw_output'])
    checks['id'] = row['id']
    compliance_results.append(checks)

compliance_df = pd.DataFrame(compliance_results).set_index('id')
display(compliance_df)

compliance_rate = compliance_df['fully_compliant'].mean()
print(f"\nFull format compliance: {compliance_rate:.1%}")
print(f"\nPer-field compliance:")
for col in ['has_label', 'has_reasoning', 'has_confidence', 'label_valid', 'confidence_valid']:
    rate = compliance_df[col].mean()
    print(f"  {col:20s}: {rate:.1%}")

print(f"\n💡 Small models often struggle with complex output formats.")
print(f"   If compliance is low, simplify the format or use a bigger model.")

> **Think About It**: You've seen that Flan-T5 doesn't always follow the
> output format we requested. In a production pipeline, a format error means
> your downstream code breaks. How would you handle this? Options include:
> retry with a simplified prompt, add a fallback parser, or use a larger model
> that follows instructions better. What are the cost/reliability tradeoffs?

## Lab 2: Build an Evaluation Framework (15 minutes)

### Your Task

Combine all four evaluation dimensions into a single scoring function,
run it on all 8 transaction outputs, and produce a summary report.

### Steps

1. **Write `evaluate_output(raw_output, actual_label)`** that returns a dict with:
   - `accuracy`: 1 if correct label, 0 if wrong
   - `format_score`: fraction of format fields present (0.0 to 1.0)
   - `label_parseable`: 1 if label could be extracted, 0 if not
   - `has_reasoning`: 1 if reasoning field present, 0 if not
2. **Run on all 8 outputs** and create a scores DataFrame
3. **Calculate overall metrics**: mean accuracy, mean format score, parseability rate
4. **Create a summary visualization** (bar chart of dimension scores)

### Expected Output

- DataFrame with per-transaction scores
- Bar chart showing average score per evaluation dimension
- Summary print: "Overall: X% accurate, Y% format-compliant, Z% parseable"

### Homework Extension

After class: extend the evaluation to detect hallucinations. Compare each
"Reasoning" field against the original transaction text — does the model
cite facts that aren't in the input? Build a simple hallucination score.

In [ ]:
# =============================================================================
# LAB 2: BUILD AN EVALUATION FRAMEWORK
# =============================================================================

def evaluate_output(raw_output, actual_label):
    """
    Evaluate a single LLM output across multiple dimensions.
    Returns a dict with scores for each dimension.
    """
    # YOUR CODE: Extract the predicted label
    predicted_label = None  # YOUR CODE

    # YOUR CODE: Check accuracy (1 if correct, 0 if wrong)
    accuracy = None  # YOUR CODE

    # YOUR CODE: Check format compliance (fraction of fields present)
    format_score = None  # YOUR CODE

    # YOUR CODE: Check if label was parseable
    label_parseable = None  # YOUR CODE

    # YOUR CODE: Check if reasoning field is present
    has_reasoning = None  # YOUR CODE

    return {
        'predicted_label': predicted_label,
        'accuracy': accuracy,
        'format_score': format_score,
        'label_parseable': label_parseable,
        'has_reasoning': has_reasoning,
    }


# YOUR CODE: Run evaluation on all outputs
scores = []  # YOUR CODE

# YOUR CODE: Create scores DataFrame
scores_df = None  # YOUR CODE

# YOUR CODE: Calculate and print overall metrics

# YOUR CODE: Create bar chart of average scores per dimension

# Verification
if scores_df is not None:
    print(f"\n✅ Evaluation framework complete!")
    display(scores_df)
else:
    print("❌ Lab 2 incomplete — fill in the YOUR CODE sections above")

# Section 3: Error Analysis & Iterative Prompt Improvement

Evaluation tells you **how well** your LLM is doing. Error analysis tells you
**why** it's failing — and what to fix.

## The Prompt Iteration Loop

```
  Start with baseline prompt
       |
  Run on test set → Collect results
       |
  Categorize errors:
  - Wrong label (misclassification)
  - Format error (didn't follow template)
  - Hallucination (invented facts)
  - Refusal (model says "I can't do this")
       |
  Identify pattern → Modify prompt
       |
  Re-run → Measure improvement
       |
  Repeat until satisfied (or diminishing returns)
```

This is **prompt engineering as a systematic discipline**, not guesswork.

In [ ]:
# =============================================================================
# DEMO: Start with a Deliberately Weak Prompt
# =============================================================================
# We'll use a vague prompt that gives poor results, then improve it.

WEAK_PROMPT = 'Is this fraud? "{description}"'


def classify_weak(description):
    """Classify with deliberately weak prompt."""
    prompt = WEAK_PROMPT.format(description=description)
    inputs = flan_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = flan_model.generate(**inputs, max_new_tokens=20, do_sample=False)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()


# Run on all transactions
weak_results = []
for txn in transaction_descriptions:
    pred = classify_weak(txn['description'])
    weak_results.append({
        'id': txn['id'],
        'actual': txn['actual_label'],
        'prediction': pred,
        'raw_response': pred
    })

weak_df = pd.DataFrame(weak_results)

# Normalize predictions
def normalize_label(pred):
    pred = pred.lower().strip()
    if 'fraud' in pred:
        return 'fraud'
    if 'legit' in pred:
        return 'legitimate'
    if pred in ['yes', 'no']:
        return 'fraud' if pred == 'yes' else 'legitimate'
    return pred

weak_df['normalized'] = weak_df['prediction'].apply(normalize_label)
weak_df['correct'] = weak_df['normalized'] == weak_df['actual']

accuracy = weak_df['correct'].mean()
print(f"Weak prompt accuracy: {accuracy:.1%}")
print()
display(weak_df[['id', 'actual', 'prediction', 'normalized', 'correct']])

In [ ]:
# =============================================================================
# DEMO: Categorize the Errors
# =============================================================================

def categorize_error(row):
    """Categorize why a prediction was wrong."""
    if row['correct']:
        return 'correct'
    if row['normalized'] not in ['fraud', 'legitimate']:
        return 'format_error'  # Model didn't give fraud/legitimate
    return 'wrong_label'  # Model gave wrong classification


weak_df['error_type'] = weak_df.apply(categorize_error, axis=1)

print("Error Analysis:")
print(f"{'='*40}")
error_counts = weak_df['error_type'].value_counts()
for err_type, count in error_counts.items():
    pct = count / len(weak_df) * 100
    icon = '✅' if err_type == 'correct' else '❌'
    print(f"  {icon} {err_type:15s}: {count} ({pct:.0f}%)")

print(f"\nDetailed errors:")
for _, row in weak_df[~weak_df['correct']].iterrows():
    print(f"  {row['id']}: predicted '{row['prediction']}' "
          f"(normalized: '{row['normalized']}'), actual '{row['actual']}'")
    print(f"    Error type: {row['error_type']}")

print(f"\n💡 Most errors are likely format errors — the model answered 'yes/no'")
print(f"   instead of 'fraud/legitimate'. The prompt was too vague!")

In [ ]:
# =============================================================================
# DEMO: Iteration 1 — Improve the Prompt Based on Error Patterns
# =============================================================================
# Error analysis showed format errors. Fix: be more explicit about expected output.

IMPROVED_PROMPT_V1 = (
    "Classify the following bank transaction as exactly 'fraud' or 'legitimate'. "
    "Respond with only one word: fraud or legitimate.\n\n"
    'Transaction: "{description}"\n\n'
    "Classification:"
)


def classify_v1(description):
    prompt = IMPROVED_PROMPT_V1.format(description=description)
    inputs = flan_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = flan_model.generate(**inputs, max_new_tokens=10, do_sample=False)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()


v1_results = []
for txn in transaction_descriptions:
    pred = classify_v1(txn['description'])
    v1_results.append({
        'id': txn['id'],
        'actual': txn['actual_label'],
        'prediction': pred
    })

v1_df = pd.DataFrame(v1_results)
v1_df['normalized'] = v1_df['prediction'].apply(normalize_label)
v1_df['correct'] = v1_df['normalized'] == v1_df['actual']

accuracy_v1 = v1_df['correct'].mean()
print(f"Iteration 1 accuracy: {accuracy_v1:.1%} (was {accuracy:.1%})")
print(f"Change: {'+' if accuracy_v1 > accuracy else ''}{(accuracy_v1 - accuracy)*100:.0f}pp")
print()
display(v1_df[['id', 'actual', 'prediction', 'normalized', 'correct']])

> **Think About It**: We improved accuracy just by clarifying the prompt.
> No retraining, no new data, no new model — just better instructions.
> This is the power (and the fragility) of prompt engineering. How would you
> version-control prompts in a production system? How do you prevent a "good"
> prompt change from breaking other edge cases?

## Lab 3: Error Analysis & Prompt Iteration (15 minutes)

### Your Task

Continue the prompt improvement loop. Starting from the v1 prompt, do
**two more iterations** to try to reach the highest accuracy possible.

### Steps

1. **Analyze v1 errors**: Which transactions does v1 still get wrong? Why?
2. **Create v2 prompt**: Fix the pattern you identified (add examples? be more specific?)
3. **Run v2** on all 8 transactions, measure accuracy
4. **Analyze v2 errors**: Any remaining failures?
5. **Create v3 prompt**: One more improvement attempt
6. **Run v3** and measure final accuracy
7. **Create an iteration summary table**: prompt version, accuracy, what changed

### Expected Output

- A DataFrame with: version, prompt_description, accuracy
- Print showing the progression from weak → v1 → v2 → v3

### Hints

- Look at WHICH transactions fail — is there a pattern?
- Try adding few-shot examples for the tricky cases
- Try adding context about what "fraud" means in banking
- Don't over-fit to 8 samples — think about what would generalize

### Homework Extension

After class: take your best prompt and run it against OpenAI GPT-4o-mini.
Does the bigger model need the same prompt improvements, or does it handle
the weak prompt just fine? What does this tell you about the relationship
between model size and prompt sensitivity?

In [ ]:
# =============================================================================
# LAB 3: ERROR ANALYSIS & PROMPT ITERATION
# =============================================================================

# --- Iteration 2 ---

# YOUR CODE: Analyze v1 errors — which transactions are still wrong?
v1_errors = None  # YOUR CODE

# YOUR CODE: Create improved prompt v2
IMPROVED_PROMPT_V2 = None  # YOUR CODE


def classify_v2(description):
    """Classify with v2 prompt."""
    # YOUR CODE
    return None  # YOUR CODE


# YOUR CODE: Run v2 on all transactions
v2_results = None  # YOUR CODE
accuracy_v2 = None  # YOUR CODE


# --- Iteration 3 ---

# YOUR CODE: Analyze v2 errors
v2_errors = None  # YOUR CODE

# YOUR CODE: Create improved prompt v3
IMPROVED_PROMPT_V3 = None  # YOUR CODE


def classify_v3(description):
    """Classify with v3 prompt."""
    # YOUR CODE
    return None  # YOUR CODE


# YOUR CODE: Run v3 on all transactions
v3_results = None  # YOUR CODE
accuracy_v3 = None  # YOUR CODE


# --- Summary ---

# YOUR CODE: Create iteration summary table
iteration_summary = None  # YOUR CODE

# Verification
if iteration_summary is not None:
    print("Prompt Iteration Summary:")
    display(iteration_summary)
    print(f"\n🎉 Lab 3 complete!")
else:
    print("❌ Lab 3 incomplete — fill in the YOUR CODE sections above")

# Summary: What We Learned Today

## Key Takeaways

### HuggingFace Local Models
| Model | Architecture | Strengths | Weaknesses |
|-------|-------------|-----------|------------|
| **GPT-2** | Decoder-only | Flexible generation | Hard to control, small |
| **Flan-T5** | Encoder-decoder | Follows instructions | Limited by size (250M) |
| **DistilBERT SST-2** | Fine-tuned classifier | Fast, consistent | Only does one task |

### LLM Evaluation — Four Dimensions
| Dimension | What to Measure | Tool |
|-----------|----------------|------|
| **Accuracy** | Correct answers vs ground truth | Label comparison |
| **Consistency** | Same input → same output | Multiple runs |
| **Format Compliance** | Follows output template | Regex parsing |
| **Hallucination** | Invents facts | Source verification |

### Error Analysis Loop
1. Run prompt on test set → measure baseline
2. Categorize errors (wrong label, format error, hallucination)
3. Identify pattern → modify prompt
4. Re-run → measure improvement
5. Repeat (usually 2-4 iterations gets you most of the gains)

## The Bigger Picture

| Week | What We Did |
|------|------------|
| 11 | LLM fundamentals — cloud APIs (OpenAI, Anthropic), basic prompting |
| **12** | **Local models, evaluation, error analysis** |
| 13 | Amazon Bedrock — managed models, synthetic data, LLM-assisted EDA |
| 14 | Training AI Models — fine-tuning with LoRA/QLoRA |

# Homework & Optional Labs

## Homework (Complete before next session)

### Homework 1: Cloud vs Local Comparison
Add OpenAI GPT-4o-mini to your Lab 1 comparison (use your Week 11 code).
Create a complete comparison table: model, accuracy, latency, estimated cost.
At what point does a local model become more cost-effective than a cloud API?

### Homework 2: Hallucination Detection
Extend the evaluation framework from Lab 2 to detect hallucinations.
For each "Reasoning" field, check if the model cites facts that aren't
in the original transaction description. Build a simple hallucination score.

### Homework 3: Prompt Regression Testing
Take your 4 prompt versions (weak, v1, v2, v3) and create a test suite.
For each prompt, record accuracy on all 8 transactions. If someone changes
a prompt in the future, they can run this suite to check for regressions.

## Optional Labs (For advanced learners)

### Optional: Automated Evaluation Metrics
Use HuggingFace `evaluate` library to compute BLEU and ROUGE scores.
See the optional notebook `week_12_optional_evaluation_metrics.ipynb`.

### Optional: Model Cards
Browse HuggingFace Model Hub and find 3 models suitable for text classification.
Read their model cards. What information helps you decide which to use?

# Great Work Today!

You've completed Week 12 of the AI for Data Scientists Academy.

**What you accomplished:**
- Ran and compared 3 different local model architectures (GPT-2, Flan-T5, DistilBERT)
- Built a structured evaluation framework (accuracy, consistency, format, hallucination)
- Performed systematic error analysis and improved prompts through iteration
- Learned the prompt engineering loop: run → evaluate → categorize errors → improve → repeat

## Coming Up Next

- **Week 13**: Amazon Bedrock — managed models, synthetic data generation, LLM-assisted EDA
- **Week 14**: Training AI Models — fine-tuning with LoRA and QLoRA
- **Weeks 15-16**: Agentic AI — ReAct agents, LangChain, multi-agent systems

## Resources

- [HuggingFace Transformers Docs](https://huggingface.co/docs/transformers/)
- [HuggingFace Model Hub](https://huggingface.co/models) — browse thousands of models
- [HuggingFace Evaluate Library](https://huggingface.co/docs/evaluate/)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)

See you in Week 13!